In [ ]:
import math
import pandas as pd

# 1. Calculate Entropy
def entropy(y):
    """
    Calculate entropy of the target column.
    """
    values = y.value_counts()
    total = len(y)

    ent = 0

    for count in values:
        probability = count / total
        ent -= probability * math.log2(probability)

    return ent

# 2. Calculate Information Gain
def information_gain(data, feature, target):
    """
    Calculate information gain for a feature.
    """

    total_entropy = entropy(data[target])

    weighted_entropy = 0

    for value in data[feature].unique():

        subset = data[data[feature] == value]

        weight = len(subset) / len(data)

        weighted_entropy += weight * entropy(subset[target])

    gain = total_entropy - weighted_entropy

    return gain


# 3. Find the Best Feature
def best_feature(data, features, target):

    gains = {}

    for feature in features:
        gains[feature] = information_gain(data, feature, target)

    best = max(gains, key=gains.get)

    return best, gains


# 4. ID3 Algorithm
def id3(data, features, target):

    # If all target values are same
    if len(data[target].unique()) == 1:
        return data[target].iloc[0]

    # If no features remain
    if len(features) == 0:
        return data[target].mode()[0]

    # Find best feature
    best, gains = best_feature(data, features, target)

    tree = {best: {}}

    remaining_features = [f for f in features if f != best]

    # Split dataset
    for value in data[best].unique():

        subset = data[data[best] == value]

        # Empty subset
        if len(subset) == 0:
            tree[best][value] = data[target].mode()[0]

        else:
            tree[best][value] = id3(
                subset,
                remaining_features,
                target
            )

    return tree


# 5. Example Dataset
data = pd.DataFrame({

    "Outlook": [
        "Sunny", "Sunny", "Overcast", "Rain",
        "Rain", "Rain", "Overcast", "Sunny",
        "Sunny", "Rain", "Sunny", "Overcast",
        "Overcast", "Rain"
    ],

    "Temperature": [
        "Hot", "Hot", "Hot", "Mild",
        "Cool", "Cool", "Cool", "Mild",
        "Cool", "Mild", "Mild", "Mild",
        "Hot", "Mild"
    ],

    "Humidity": [
        "High", "High", "High", "High",
        "Normal", "Normal", "Normal", "High",
        "Normal", "Normal", "Normal", "High",
        "Normal", "High"
    ],

    "Wind": [
        "Weak", "Strong", "Weak", "Weak",
        "Weak", "Strong", "Strong", "Weak",
        "Weak", "Weak", "Strong", "Strong",
        "Weak", "Strong"
    ],

    "Play Tennis": [
        "No", "No", "Yes", "Yes",
        "Yes", "No", "Yes", "No",
        "Yes", "Yes", "Yes", "Yes",
        "Yes", "No"
    ]
})


# 6. Train ID3
features = ["Outlook", "Temperature", "Humidity", "Wind"]
target = "Play Tennis"

tree = id3(data, features, target)

print("Decision Tree:")
print(tree)

Decision Tree:
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}
